# Sequential Shattering Capacity — $k = m$ sweep

For each value $m \in \{1, 2, \ldots, M_{\max}\}$ we run the **sequential capacity estimation** with center length $k = m$ (same length as the query series).  
Hard-DTW validation is **enabled**: a witness is only accepted if strict hard-DTW separation holds.


## 1 · Imports & Configuration


In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath("soft_dtw_solver.py")))

from soft_dtw_solver import sequential_capacity_estimation

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Shared hyperparameters ────────────────────────────────────────────────────
GAMMA              = 1.0   # Soft-DTW smoothing parameter
EPOCHS             = 500   # gradient steps per optimisation call
RETRIES            = 5     # retries per subset (optimize_ball_robust)
MAX_RETRIES_STEP4  = 10    # retries when trying to add a new point to the set
MAX_D              = 20    # upper bound on shattering dimension to search for
NUM_RUNS           = 1     # independent greedy runs per (k, m) configuration
VALIDATION         = True  # require hard-DTW strict separation

# ── Values of m (= k) to sweep ───────────────────────────────────────────────
M_VALUES = [1, 2, 3, 4, 5]   # add larger values if runtime permits


## 2 · Run Sequential Shattering Estimation for each $k = m$

For each value of $m$, `sequential_capacity_estimation` greedily adds random query series of length $m$ until it can no longer shatter the current set.  The center length is set to $k = m$.

> ⚠️ Each configuration can take several minutes. Increase `MAX_D` or `M_VALUES` carefully.


In [ ]:
results = []   # list of dicts, one per (k=m) configuration

for m in M_VALUES:
    k = m   # k = m constraint
    print(f"\n{'='*60}")
    print(f"  Running  m = {m},  k = {k}")
    print(f"{'='*60}")

    out_csv = f"witnesses_k{k}_m{m}.csv"

    if NUM_RUNS == 1:
        d_max, X_final, witnesses = sequential_capacity_estimation(
            m=m, k=k,
            gamma=GAMMA,
            max_retries_step4=MAX_RETRIES_STEP4,
            max_d=MAX_D,
            witness_csv_path=out_csv,
            num_runs=NUM_RUNS,
            verbose=False,
            validation=VALIDATION,
        )
        all_d = [d_max]
    else:
        d_max, X_final, witnesses, all_d = sequential_capacity_estimation(
            m=m, k=k,
            gamma=GAMMA,
            max_retries_step4=MAX_RETRIES_STEP4,
            max_d=MAX_D,
            witness_csv_path=out_csv,
            num_runs=NUM_RUNS,
            verbose=False,
            validation=VALIDATION,
        )

    results.append({
        "m": m,
        "k": k,
        "d_max": d_max,
        "all_d": all_d,
        "X_final": X_final,
        "witnesses": witnesses,
    })
    print(f"  → d_max = {d_max}  (all runs: {all_d})")

print("\nAll configurations done.")


## 3 · Results Table


In [ ]:
rows = []
for r in results:
    all_d = r["all_d"]
    rows.append({
        "k = m":        r["m"],
        "d_max (best)": r["d_max"],
        "d_max (mean)": round(np.mean(all_d), 2),
        "d_max (all runs)": str(all_d),
        "hit MAX_D":    "✓" if r["d_max"] >= MAX_D else "",
    })

df = pd.DataFrame(rows).set_index("k = m")
print(df.to_string())
df


## 4 · Plot: $d_{\max}$ vs. $k = m$


In [ ]:
m_vals  = [r["m"]     for r in results]
d_vals  = [r["d_max"] for r in results]
hit_max = [r["d_max"] >= MAX_D for r in results]

fig, ax = plt.subplots(figsize=(7, 4))

# All points
ax.plot(m_vals, d_vals, marker="o", color="steelblue", linewidth=1.5, label="$d_{\\max}$")

# Mark configurations that hit MAX_D (lower bound — may be higher)
if any(hit_max):
    ax.scatter(
        [m for m, h in zip(m_vals, hit_max) if h],
        [d for d, h in zip(d_vals,  hit_max) if h],
        marker="^", s=120, color="tomato", zorder=5,
        label=f"hit MAX\\_D={MAX_D} (lower bound)",
    )

# Reference line at MAX_D
ax.axhline(MAX_D, color="tomato", linestyle="--", linewidth=0.8, alpha=0.6, label=f"MAX\\_D = {MAX_D}")

ax.set_xlabel("$k = m$  (query length = center length)", fontsize=12)
ax.set_ylabel("Estimated shattering dimension $d_{\\max}$", fontsize=12)
ax.set_title("Sequential shattering capacity with $k = m$", fontsize=13)
ax.set_xticks(m_vals)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
